# 02 - Model training, evaluation & cost projection

Train the LightGBM TEU forecaster with a time-based holdout, compare it with a seasonal-naive
baseline, inspect feature importance, then turn the forecast into an RM haulage budget with
the rate card. Set `use_mlflow=True` to log the run (then `mlflow ui`).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

In [ ]:
import pandas as pd

from src.forecasting.data import load_volumes
from src.forecasting.predict import DemandForecaster
from src.forecasting.train import train

volumes = load_volumes(ROOT / "data/raw/container_volumes.csv")
result = train(
    volumes, model_path=ROOT / "models/volume_model.joblib", holdout_days=56, tune=True, use_mlflow=False
)
pd.DataFrame({"LightGBM": result.metrics, "Seasonal naive": result.baseline_metrics}).round(3)

## Feature importance

In [ ]:
model = result.bundle["model"]
pd.Series(model.feature_importances_, index=result.bundle["features"]).sort_values(ascending=False)

## Recursive 30-day forecast

In [ ]:
forecaster = DemandForecaster.load(result.model_path)
fc = forecaster.forecast("palm-oil", horizon_days=30)
fc.head(10)

## From TEU to ringgit
Price one 20ft palm-oil container to Pasir Gudang with the rate card, then scale by the forecast.

In [ ]:
from src.costing.quote import QuoteRequest, build_quote, compare_modes
from src.costing.rates import RateCard

rates = RateCard.load(ROOT / "data/reference/rates.json")
quote = build_quote(QuoteRequest(destination="Pasir Gudang", container_size="20", cargo_type="palm-oil"), rates)
pd.DataFrame([i.model_dump() for i in quote.line_items])

In [ ]:
containers = round(fc["forecast"].sum())
print(f"Forecast: {containers} TEU over 30 days")
print(f"Projected cost:    RM{quote.cost_rm * containers:,.0f}")
print(f"Projected revenue: RM{quote.price_rm * containers:,.0f}")
print(f"Projected margin:  RM{quote.margin_rm * containers:,.0f}")
print(f"Projected diesel:  {quote.diesel_litres * containers:,.0f} L")

## Truck vs rail to Penang

In [ ]:
cmp = compare_modes(QuoteRequest(destination="Penang", container_size="40", cargo_type="chemicals"), rates)
print(f"Cheapest: {cmp.cheapest_mode.value}, fastest: {cmp.fastest_mode.value}, premium per day saved: RM{cmp.premium_rm_per_day_saved}")
pd.DataFrame([{"mode": q.mode.value, "price_rm": q.price_rm, "transit_days": cmp.transit_days[q.mode.value]} for q in cmp.quotes])